<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/appendix/B_mcp_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Appendix B：W&B MCP サーバー

W&B の公式 MCP（Model Context Protocol）サーバーを使うと、
Claude Code・Cursor・Claude Desktop などの AI エージェントが
自然言語で W&B / Weave のデータに直接アクセスできます。

**リポジトリ:** https://github.com/wandb/wandb-mcp-server
**ホスト済みエンドポイント:** `https://mcp.withwandb.com/mcp`（ゼロコンフィグ）

## 学習内容
1. MCP サーバーの概要と利用可能なツール
2. Claude Code への接続（ホスト済み / ローカル）
3. Claude Desktop への接続
4. Dedicated Cloud 向け設定
5. 各ツールの使い方と実用例


In [ ]:
# ローカル環境: uv sync で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "python-dotenv>=1.0.0"

In [ ]:
import os

# Colab / ローカル環境の自動判定
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
    print("Google Colab 環境")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv; load_dotenv()
    # ローカル: .env に空欄で書かれていた場合も削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)
    print("ローカル環境")

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")

## 1. MCP サーバーの概要

### 利用可能な 6 つのツール

| ツール名 | 機能 |
|---|---|
| `query_wandb_tool` | GraphQL で run・メトリクス・実験を照会 |
| `query_weave_traces_tool` | LLM トレースと評価結果を分析 |
| `count_weave_traces_tool` | トレース数・ストレージ容量をカウント |
| `create_wandb_report_tool` | W&B レポートをプログラムで生成 |
| `query_wandb_entity_projects` | エンティティのプロジェクト一覧を取得 |
| `query_wandb_support_bot` | W&B ドキュメント・wandbot を参照 |

### 接続方式

```
ホスト済み（推奨）: https://mcp.withwandb.com/mcp
                    ← ゼロコンフィグ、常に最新版
ローカル（STDIO）:  uvx wandb_mcp_server
                    ← オフライン・Dedicated Cloud 環境向け
ローカル（HTTP）:   uvx wandb_mcp_server --transport http --port 8080
                    ← セルフホスト
```


## 2. Claude Code への接続

### ホスト済みエンドポイント（最もシンプル）

ターミナルで以下を実行します（**このノートブックの外で実行**）:

```bash
claude mcp add --transport http wandb https://mcp.withwandb.com/mcp \
  --scope user \
  --header "Authorization: Bearer $WANDB_API_KEY"
```

### Dedicated Cloud 環境の場合

`WANDB_BASE_URL` を環境変数として設定します:

```bash
# ローカル STDIO モードで Dedicated Cloud に接続
claude mcp add wandb \
  uvx -- --from git+https://github.com/wandb/wandb-mcp-server wandb_mcp_server \
  --scope user \
  --env WANDB_API_KEY=$WANDB_API_KEY \
  --env WANDB_BASE_URL=https://your-org.wandb.io
```

### 接続確認

Claude Code で以下のように話しかけます:
```
「W&B の weave-course プロジェクトにあるトレースの数を教えてください」
```

→ MCP サーバーが `count_weave_traces_tool` を呼び出して回答します。


## 3. Claude Desktop への接続

`~/Library/Application Support/Claude/claude_desktop_config.json`（macOS）を編集:

### ホスト済みエンドポイント

```json
{
  "mcpServers": {
    "wandb": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "https://mcp.withwandb.com/mcp",
        "--header",
        "Authorization: Bearer YOUR_WANDB_API_KEY"
      ]
    }
  }
}
```

### ローカル STDIO（Dedicated Cloud 対応）

```json
{
  "mcpServers": {
    "wandb": {
      "command": "uvx",
      "args": [
        "--from",
        "git+https://github.com/wandb/wandb-mcp-server",
        "wandb_mcp_server"
      ],
      "env": {
        "WANDB_API_KEY": "your_api_key",
        "WANDB_BASE_URL": "https://your-org.wandb.io"
      }
    }
  }
}
```

設定後、Claude Desktop を再起動すると 🔨 ツールアイコンが表示されます。


## 4. 各ツールの使い方と実用例

MCP サーバーに接続した AI エージェントは、以下のような自然言語の質問に答えられます。
それぞれのツールの「使い方の例」と「エージェントへの指示文例」を示します。


### 4.1 `query_wandb_entity_projects` — プロジェクト一覧

**用途:** エンティティに紐づくプロジェクトの一覧を取得します。

**エージェントへの指示例:**
```
「私の W&B エンティティ（your-entity）にあるプロジェクトを全部見せてください」
```

**エージェントの動作:**
1. `query_wandb_entity_projects` を呼び出し
2. プロジェクト名・作成日・run 数などを表形式で返す


### 4.2 `query_wandb_tool` — 実験データの照会

**用途:** GraphQL で W&B の run・メトリクス・設定を照会します。

**エージェントへの指示例:**
```
「weave-course プロジェクトの直近 10 件の学習 run を、
 最終損失の低い順に並べて表にしてください」
```

```
「art-training という job_type を持つ run の中で、
 avg_reward が最も高かった run の設定を教えてください」
```

**エージェントの動作:**
1. `query_wandb_tool` に GraphQL クエリを組み立てて送信
2. run の設定・メトリクスを取得してフォーマット


### 4.3 `query_weave_traces_tool` — Weave トレースの分析

**用途:** LLM アプリケーションのトレース・評価結果を照会・分析します。

**エージェントへの指示例:**
```
「weave-course プロジェクトの run_agent というオペレーションのトレースを調べて、
 平均レイテンシとトークン使用量を教えてください」
```

```
「直近の Evaluation.evaluate の結果を取得して、
 exact_match スコアが最も低かった入力例を 3 件見せてください」
```

**エージェントの動作:**
1. `query_weave_traces_tool` でトレースをフィルタリング
2. 統計量を計算して返す


### 4.4 `count_weave_traces_tool` — トレース数のカウント

**用途:** トレースの件数やストレージ容量を効率的に取得します。

**エージェントへの指示例:**
```
「weave-course プロジェクトに今日ログされたトレースは何件ありますか？」
```

```
「エラーになったトレースは全体の何%ですか？」
```

**ポイント:** 全トレースを取得せずにサーバー側でカウントするため、
大量トレースがあっても高速に応答します。


### 4.5 `create_wandb_report_tool` — レポートの自動生成

**用途:** W&B レポート（ダッシュボード）をプログラムで生成します。

**エージェントへの指示例:**
```
「weave-course プロジェクトの評価結果をまとめた W&B レポートを作成してください。
 各モデルバージョンの exact_match スコアの推移グラフを含めてください」
```

**エージェントの動作:**
1. `query_wandb_tool` で必要なデータを収集
2. `create_wandb_report_tool` でレポートを生成
3. レポートの URL を返す


### 4.6 `query_wandb_support_bot` — ドキュメント参照

**用途:** W&B の公式ドキュメントや wandbot に質問します。

**エージェントへの指示例:**
```
「weave.EvaluationLogger の使い方を教えてください」
```

```
「W&B Sweeps でベイズ最適化を使う方法は？」
```

→ 常に最新ドキュメントを参照するため、新機能にも対応しています。


## 5. MCP を使った実際のワークフロー例

以下は、Claude Code + W&B MCP サーバーを使った典型的な分析セッションです。

```
ユーザー: 「weave-course プロジェクトのトレースを分析して、
           どのツールが最も頻繁に呼ばれているか教えてください」

Claude:  [query_weave_traces_tool を呼び出し]
          → get_weather: 47回
          → calculate:   38回  ← 最多
          → search_web:  21回

         calculate ツールが最も頻繁に呼ばれており、
         平均レイテンシは 12ms です。

ユーザー: 「このデータを使って W&B レポートを作成してください」

Claude:  [create_wandb_report_tool を呼び出し]
          レポートを作成しました: https://wandb.ai/.../reports/...
```

### Dedicated Cloud でのポイント

- `WANDB_BASE_URL` を必ず設定する
- ホスト済みエンドポイントは Public Cloud 向け。Dedicated Cloud では **ローカル STDIO モード**を使用
- セキュリティ上、API キーは環境変数経由で渡す（コマンドライン引数に直接書かない）


## 6. wandb/skills と MCP サーバーの使い分け

| | wandb/skills | W&B MCP サーバー |
|---|---|---|
| **主な用途** | エージェントが W&B/Weave のコードを書く精度を上げる | エージェントが W&B/Weave のデータに直接アクセスする |
| **動作原理** | スキルファイルをコンテキストに読み込む | MCP プロトコルでツールを呼び出す |
| **インストール** | `npx skills add wandb/skills` | `claude mcp add wandb ...` |
| **向いている場面** | コード生成・デバッグ支援 | データ分析・レポート生成・質問応答 |
| **組み合わせ** | ✅ 両方同時に使うと相乗効果 | ✅ 両方同時に使うと相乗効果 |

### 両方を組み合わせた理想的なセットアップ

```bash
# 1. skills をインストール（コード生成精度の向上）
npx skills add wandb/skills

# 2. MCP サーバーを接続（データアクセス）
claude mcp add --transport http wandb https://mcp.withwandb.com/mcp \
  --scope user \
  --header "Authorization: Bearer $WANDB_API_KEY"
```

→ エージェントは「正確なコードを書く知識」と「リアルタイムデータへのアクセス」
  を両方持つことになり、W&B / Weave に関するあらゆるタスクを高精度で実行できます。


## まとめ

| 機能 | URL/コマンド |
|---|---|
| wandb/skills インストール | `npx skills add wandb/skills` |
| MCP ホスト済みエンドポイント | `https://mcp.withwandb.com/mcp` |
| MCP GitHub リポジトリ | https://github.com/wandb/wandb-mcp-server |
| Weave ドキュメント | https://docs.wandb.ai/weave |
